# Spreadsheet Merger

Upload multiple Excel files and combine them into a single spreadsheet.

**How it works:**
1. Run the cells in order (Runtime → Run all, or Shift+Enter through each cell).
2. When prompted, select multiple `.xlsx`/`.xls`/`.csv` files at once (ctrl/cmd-click or shift-click in the file picker).
3. The script checks that every file has the same column names (order doesn't matter). If any file's columns don't match, it tells you exactly which columns are missing/extra before combining anything.
4. Once validated, all rows are stacked into one sheet and a `source_file` column is added so you can trace each row back to its original file.
5. The merged file downloads automatically as `merged_spreadsheet.xlsx`.

In [ ]:
# Step 1: Upload files
from google.colab import files

print("Select all the spreadsheet files you want to combine (you can multi-select).")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No files were uploaded.")

print(f"\nUploaded {len(uploaded)} file(s):")
for name in uploaded:
    print(f"  - {name}")

In [ ]:
# Step 2: Read every file and validate that columns match
import io
import pandas as pd

SHEET_NAME = 0  # change this if you need a specific sheet name/index from each workbook

def read_spreadsheet(filename, raw_bytes):
    buffer = io.BytesIO(raw_bytes)
    if filename.lower().endswith(".csv"):
        return pd.read_csv(buffer)
    return pd.read_excel(buffer, sheet_name=SHEET_NAME)

dataframes = {}
for filename, raw_bytes in uploaded.items():
    dataframes[filename] = read_spreadsheet(filename, raw_bytes)

# Use the first uploaded file as the reference column set
reference_name = next(iter(dataframes))
reference_columns = set(dataframes[reference_name].columns)

mismatches = {}
for filename, df in dataframes.items():
    columns = set(df.columns)
    missing = reference_columns - columns
    extra = columns - reference_columns
    if missing or extra:
        mismatches[filename] = {"missing": missing, "extra": extra}

if mismatches:
    print(f"Column mismatch detected (using '{reference_name}' as the reference):\n")
    for filename, diff in mismatches.items():
        print(f"{filename}:")
        if diff["missing"]:
            print(f"  missing columns: {sorted(diff['missing'])}")
        if diff["extra"]:
            print(f"  unexpected columns: {sorted(diff['extra'])}")
    raise ValueError(
        "Not all files have matching columns. Fix the files above (or re-upload) and re-run."
    )
else:
    print("All files have matching columns. Safe to combine.")

In [ ]:
# Step 3: Combine into a single spreadsheet
combined_frames = []
for filename, df in dataframes.items():
    df = df[list(dataframes[reference_name].columns)]  # enforce a consistent column order
    df = df.copy()
    df.insert(0, "source_file", filename)
    combined_frames.append(df)

combined_df = pd.concat(combined_frames, ignore_index=True)
print(f"Combined {len(dataframes)} files into {len(combined_df)} total rows.")
combined_df.head()

In [ ]:
# Step 4: Save and download the merged spreadsheet
output_filename = "merged_spreadsheet.xlsx"
combined_df.to_excel(output_filename, index=False)

files.download(output_filename)
print(f"Saved and downloading '{output_filename}'.")